In [ ]:
                                              # CHAPTER-8 : Semantic Searchand Retrieval-AugmentedGeneration

In [ ]:
#                ======= Dense retrieval=========

In [ ]:
# Install the latest versions of cohere
!pip install -U cohere

In [ ]:
# Unistall the version you have
!pip uninstall -y cohere

# used to install the specific version
!pip install cohere==4.57

In [ ]:
# Coherent client

In [ ]:
# cohere = it is a python library that allows us to communicate with cohere AI
# It is used to generate embedding
import cohere

# NumPy used for numbers and arrays calculations
import numpy as np

# Pandas library used to work with data in tables
import pandas as pd

# used to display the progress bar
from tqdm import tqdm


# Paste your API key here. Remember to not share publicly

# api key = it is like a password . It helps to access the cohere ai serives
api_key = 'API_KEY'

# creating cohere client so we can use the cohere ai with our key
# Create and retrieve a Cohere API key from os.cohere.ai
co = cohere.Client(api_key)

In [ ]:
# Text chunking

In [ ]:
# Information like a document
text = """
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades"""

# Splits the text into sentences where . appears it beraks there
texts = text.split('.')

# for every sentence in texts variable it removes the unwanted spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [ ]:
# Ebedding the text chunks

In [ ]:
# co = previously we created the cohere client
# co.embed = here we are telling that use these things and give me the embeddings
response = co.embed(

        # embed the texts
    texts=texts,

        # use this model for embeddings
    model="embed-english-v3.0",

        # it is a paramater of cohere that the role of text
        # we are telling that this texts will be used later for searching puprose
    input_type="search_document",

).embeddings # cohere retruns the result including embedding , metadata and other info
# this tells to take only embeddings

# convert the embeddings into numpy array
embeds = np.array(response)

# Shape of the embedding vector
print(embeds.shape)

In [ ]:
# FIASS = Facebook AI Similarity search
# Helps to search the similairty of vectors
!pip install faiss-cpu

In [ ]:
# Building the search index

In [ ]:
# Importing FAISS library
import faiss

# in shape we have 2 dimensions
# 0 = No.of.texts
# 1 = No.of.values in embedding vector
dim = embeds.shape[1]

# creating the faiss index to store the values later
# this tells to create the index for the vector which contains dim values in each vector
index = faiss.IndexFlatL2(dim)

# Checking the index is ready or not
print(index.is_trained)

# Faiss accpets the vectors in a specific numerical format(32 bit)
# Convert the embedding vectors into floating array numbers
# add these to faiss index
index.add(np.float32(embeds))


In [ ]:
# Seraching the indexes

In [ ]:
# Function has 2 paramters
# query=users search question
# number_of_results = how many search results does the model want to return
def search(query, number_of_results=3):

    # 1. Get the query's embedding
    query_embed = co.embed(

         # use the list that contains the query for generating embeddings
        texts=[query],

         #  model name for using
        model="embed-english-v3.0",

         # Tells this was a search query
        input_type="search_query"

        # take the first embedding vector from returned results
    ).embeddings[0]


    """ 2. Retrieve the nearest neighbors
     search the faiss index using the query and return the required no.of.results
     distances = This contains the distances between the query and the closest vectors.
     similar_item_ids = faiss tells the which vectors are close
     IDs help us find the original text        """
    distances, similar_item_ids = index.search(

        # convert the query vector into 32 bit floating point numbers which is expected by faiss
        np.float32([query_embed]),
        number_of_results
    )

    # texts = python list , convert it into array
    # 3. Format the results
    texts_np = np.array(texts)

  # # Create a DataFrame to store the search results
    results = pd.DataFrame(
        data={
             # Get the actual text of the documents found by FAISS using their indexes
            'texts': texts_np[similar_item_ids[0]],

             # Get the distance score for each retrieved document
            'distance': distances[0]
        }
    )


    # Print the user's search query and a heading for the retrieved results
    print(f"Query: '{query}'\n\nNearest neighbors:")

# Return the DataFrame containing the retrieved texts and their distance scores
    return results

In [ ]:
query = "how precise was the science"
results = search(query)
print(results)

In [ ]:
#                  ======  Keyword Search  by bm25  ============

In [ ]:
!pip install rank-bm25

In [ ]:
# Import BM25Okapi for keyword-based text ranking
from rank_bm25 import BM25Okapi

# Import the built-in English stop words
from sklearn.feature_extraction import _stop_words

# used for handling punctuation
import string

In [ ]:
# BM25 tokenizer

In [ ]:
# creating a function
def bm25_tokenizer(text):

  # create an ampty list
    tokenized_doc = []

# convert the text into lowercase and split the sentence into individual words
    for token in text.lower().split():

      # removes the punctuation from beginning and end of word
        token = token.strip(string.punctuation)

        # length of token must be > 0 if "?" after removing punctutaion token becomes empty
        # checking the token is stop word or not
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:

          # if not a stop word then add the token to our tokenized_doc list
            tokenized_doc.append(token)


# return the list
    return tokenized_doc

In [ ]:
# Document (Information) tokenizer

In [ ]:
from tqdm import tqdm
# stores the cleaned texts / doc
tokenized_corpus = []

# go through the each sentence in the texts/doc remove stop words and add cleaned texts to tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

100%|██████████| 15/15 [00:00<00:00, 19082.37it/s]


In [ ]:
# BM25 model

In [ ]:
# creating bm25 object to create
# bm25 search system using the tokenized documents
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
# Search function

In [ ]:
import numpy as np

# top_k = how many results you want to display
# num_candidates = how many sentences you want to consider for search
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    # we have cleaned list query in bm25 we have cleaned list of documents
    # check how relavent every document to the query and retrun the score
    # high score = more relavent
    bm25_scores = bm25.get_scores(
        bm25_tokenizer(query)
    )

    # Get top candidate document IDs
       # np.argparition = helpa to find the indexes of the high score documents
       # np.argpartition() rearranges the array so that the top 3 candidate indexes are placed at the end (not sorted)
    top_n = np.argpartition(

          # we have score of every document
        bm25_scores,

          # this tells to take the n top results
        -num_candidates
    )[-num_candidates:]  # take the last 3 positions cause it have high score doc


    # This having selected document's indexes and scores
    bm25_hits = [
        {
            'corpus_id': idx,     # stores the id of doc
            'score': bm25_scores[idx]  # stores the score
        }
        for idx in top_n  # Goes through each document index stored in top_n
    ]

    # Sort results by score (highest first)
    bm25_hits = sorted(

       # we have top selected doc with index and scores
        bm25_hits,

       # x= one document , Gets the score value from that item
        key=lambda x: x['score'],

       # sort in descending order
        reverse=True
    )

    print("Top-3 lexical search (BM25) hits")

    # Print top-k results
    #Takes the first top_k results from the sorted bm25_hits list
    for hit in bm25_hits[0:top_k]:
        print(

              # t → adds a tab space
              # {:.3f} → displays the BM25 score with 3 decimal places
              # {} → displays the document text
              # .format(...) → puts the actual values into {:.3f} and {}
            "\t{:.3f}\t{}".format(

              # Gets the BM25 score of the current result
                hit['score'],

              # Uses the document's ID/index to get the original document text.
              # Replaces new-line characters with spaces so the document is printed in one line
                texts[hit['corpus_id']].replace("\n", " ")
            )
        )

In [ ]:
# call the function
keyword_search(query = "how precise was the science")


In [ ]:
                 # RERANKING

In [ ]:
query = "how precise was the science"
# The reranker model directly evaluates and ranks the documents
# Send the query and all documents to Cohere's reranking model
results = co.rerank(query=query, documents=texts, top_n=3,

   # Include the actual document text in the returned results
return_documents=True
)

# Get the list of reranked results from the response
results.results

In [ ]:
# enumerate() gives us two things for each result:
# idx = the position/index of the result
# result = the actual result object

# Go through each reranked result and print its position, relevance score, and actual document text.
for idx, result in enumerate(results.results):
  print(idx, result.relevance_score , result.document.text)

In [ ]:
#  KEYWORD AND RERANKING
# BM25 also produces a top 3 for display, but those are not the final results.
# The reranker takes the 10 BM25 candidates and chooses a new final top 3.

In [ ]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    # 1. BM25 search
    bm25_scores = bm25.get_scores(
        bm25_tokenizer(query)
    )

    # 2. Get top candidate document IDs
    top_n = np.argpartition(
        bm25_scores,
        -num_candidates
    )[-num_candidates:]

    # 3. Create BM25 hits
    bm25_hits = [
        {
            'corpus_id': idx,
            'score': bm25_scores[idx]
        }
        for idx in top_n
    ]

    # 4. Sort BM25 results
    bm25_hits = sorted(
        bm25_hits,
        key=lambda x: x['score'],
        reverse=True
    )

    # 5. Print top BM25 results
    print("Top-3 lexical search (BM25) hits")

    for hit in bm25_hits[0:top_k]:
        print(
            "\t{:.3f}\t{}".format(
                hit['score'],
                texts[hit['corpus_id']].replace("\n", " ")
            )
        )

    # 6. Get documents for reranking
    # Extract the original text of all BM25 candidate documents
    docs = [
        texts[hit['corpus_id']]
        for hit in bm25_hits
    ]

    # 7. Rerank using Cohere
    # Print a heading showing that BM25 results will now be reranked
    print(
        f"\n\nTop-{top_k} hits by rank-API "
        f"({len(bm25_hits)} BM25 hits re-ranked)"
    )

   # Send the query and BM25 candidate documents to Cohere's reranker
    results = co.rerank(
        query=query,
        documents=docs,
        top_n=top_k,
        return_documents=True
    )

    # 8. Print reranked results
    # Go through each result returned by the Cohere reranker
    for hit in results.results:
        print(

              # Print the rerank relevance score and the corresponding document text
            "\t{:.3f}\t{}".format(
                hit.relevance_score,
                hit.document.text.replace("\n", " ")
            )
        )

In [ ]:
keyword_and_reranking_search(query = "how precise was the science")

In [ ]:
 #    ===== Retrieval-Augmented Generation (RAG) ====

In [ ]:
query = "income generated"
# 1- Retrieval
# We'll use embedding search. But ideally we'd do hybrid
# Search the document collection using embedding search
# The search function returns the documents most relevant to the query
results = search(query)


# 2- Grounded Generation
# Convert the retrieved document texts into dictionaries
# because the co.chat() function expects documents in this format
docs_dict = [{'text': text}
for text in results['texts']]

# Send the user's question and the retrieved documents to the LLM
# The LLM uses the retrieved documents as context to generate the answer
response = co.chat(
message = query,
documents=docs_dict
)

print(response.text)


In [ ]:
#     RAG with Local Models

In [ ]:
# Loading the generation model
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

In [ ]:
# Import LlamaCpp to load and use a local GGUF language model with LangChain
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
model_path="Phi-3-mini-4k-instruct-fp16.gguf",
n_gpu_layers=-1,
max_tokens=500,
n_ctx=2048,
seed=42,
verbose=
False
)

In [ ]:
# Import Hugging Face embedding model support from LangChain
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(
model_name='thenlper/gte-small'
)

In [ ]:
# Import FAISS vector store from LangChain
# FAISS is used to store and search text embeddings efficiently
from langchain.vectorstores import FAISS

# Create a local FAISS vector database
# texts = the list of documents/sentences we want to store
# embedding_model = converts each text into a numerical embedding
# FAISS stores these embeddings so we can later search for similar documents
db = FAISS.from_texts(texts, embedding_model)

In [ ]:
from langchain import PromptTemplate
# Create a prompt template
template = """<|user|>
Relevant information:
{context}
Provide a concise answer the following question using the
relevant information provided above:
{question}<|end|>
<|assistant|>"""

# Create a PromptTemplate object using the template above
prompt = PromptTemplate(
template=template,
input_variables=["context", "question"]
)

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community
!pip install langchain==0.0.353
!pip install llama-cpp-python

In [ ]:
# Import RetrievalQA, which helps create a RAG question-answering pipeline
from langchain.chains import RetrievalQA


# RAG pipeline
rag = RetrievalQA.from_chain_type(
llm=llm,

   # "stuff" means all retrieved documents are placed together
    # into the prompt as context before sending them to the LLM
chain_type='stuff',

    # Convert the FAISS database into a retriever
    # The retriever searches the vector database and finds
    # documents that are relevant to the user's question
retriever=db.as_retriever(),

chain_type_kwargs={
"prompt": prompt
},
    # Show the intermediate steps and information while the RAG pipeline runs
verbose=True
)

In [ ]:
# Question ==> Retrieve relevant documents ==> Add them to prompt ==> LLM generates answer.
rag.invoke('Income generated')